# 06 - Enhancements

三个增强实验：
1. **P1: HumanEval Benchmark** — 公认基准评测，获取 pass@1 指标
2. **P2: Coder 工具调用优化** — 减少冗余调用，降低 MCP 模式 token 消耗
3. **P3: 多模型对比** — DeepSeek vs Qwen 对比

**预估总耗时**: ~1-1.5 小时
**预估总 Token**: ~500K-800K

In [ ]:
# Cell 1: 环境准备
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/CodeAgent-MCP'
os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

!pip install -q openai mcp pydantic pyyaml rich nest_asyncio

In [ ]:
# Cell 2: API Keys
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')

# SiliconFlow key for P3 multi-model comparison (if available)
try:
    os.environ['SILICONFLOW_API_KEY'] = userdata.get('SILICONFLOW_API_KEY')
    print('SiliconFlow key loaded')
except:
    print('SiliconFlow key not set (P3 will be skipped)')

with open('/tmp/.api_key', 'w') as f:
    f.write(os.environ['OPENAI_API_KEY'])

print('API keys configured')

---
## P1: HumanEval Benchmark

用 OpenAI HumanEval 数据集评测 pass@1。

- 164 个函数补全任务，自动执行测试验证正确性
- 使用 single agent (Coder only) 模式，更公平地与其他系统对比
- DeepSeek-chat 的 HumanEval 参考水平: ~70-80% (取决于版本)

In [ ]:
# Cell 3: 下载 HumanEval 数据集
!pip install -q human-eval

import json
from human_eval.data import read_problems

problems = read_problems()
print(f'Loaded {len(problems)} HumanEval problems')

# 转为 JSONL 格式保存
output_path = 'eval/humaneval_problems.jsonl'
with open(output_path, 'w', encoding='utf-8') as f:
    for task_id, problem in sorted(problems.items()):
        problem['task_id'] = task_id
        f.write(json.dumps(problem, ensure_ascii=False) + '\n')

print(f'Saved to {output_path}')
# 展示一个样例
sample = list(problems.values())[0]
print(f"\nSample: {sample['entry_point']}")
print(sample['prompt'][:200])

In [ ]:
# Cell 4: 运行 HumanEval (single agent mode)
# 先跑 20 题测试，确认无误后再跑全量
%%writefile /tmp/run_humaneval.py
import asyncio
import json
import os
import sys
import time

os.chdir('/content/drive/MyDrive/CodeAgent-MCP')
sys.path.insert(0, '.')

with open('/tmp/.api_key') as f:
    os.environ['OPENAI_API_KEY'] = f.read().strip()

from eval.humaneval_adapter import main
asyncio.run(main())

In [ ]:
# Cell 5: 先跑 20 题验证
!python /tmp/run_humaneval.py --limit 20 --provider default --mode single

In [ ]:
# Cell 6: 确认无误后，跑全量 164 题
# 预估耗时 ~30-40 分钟
!python /tmp/run_humaneval.py --provider default --mode single

---
## P2: Coder 工具调用优化

优化 Coder prompt，减少 MCP 模式下的冗余工具调用（重复 file_list/file_read），
然后重跑 MCP benchmark 对比 token 消耗变化。

**实验设计**: 选 3 个代表性任务 (B1, B5, B6)，对比优化前后的 token 消耗。

In [ ]:
# Cell 7: 备份当前 Coder prompt，应用优化版
import yaml

with open('config/agents.yaml', 'r', encoding='utf-8') as f:
    agents_config = yaml.safe_load(f)

# 保存原始 prompt
original_coder_prompt = agents_config['coder']['system_prompt']
print('Original Coder prompt:')
print(original_coder_prompt)

# 优化版 prompt: 更明确的工具调用规则
optimized_coder_prompt = '''你是一个高级Python开发工程师。根据任务描述编写高质量的 Python 代码，包含类型注解，遵循 PEP 8。

当你有工具可用时，严格按以下顺序操作：
第1步：如果 workspace 有已有文件，file_read 需要参考的文件（每个文件只读一次）
第2步：用 file_write 写入主源码文件（一次调用写完整个文件）
第3步：用 file_write 写入测试文件
第4步（可选）：用 shell_exec 运行 pytest 验证

工具调用效率规则：
- 不要在空目录调用 file_list 或 file_search
- 不要写完文件后再 file_read 自己刚写的文件
- 不要多次小片段追加写入，一次 file_write 写完整个文件
- 不要重复读取同一个文件，读一次后记住内容
- 不要用 file_search 搜索你刚创建的文件
- 已知 workspace 为空时，直接跳到 file_write

完成后用一句话说明做了什么，不要重复输出代码内容。
'''

print('\n' + '='*60)
print('Optimized Coder prompt:')
print(optimized_coder_prompt)

In [ ]:
# Cell 8: 对比实验 (B1, B5, B6)
%%writefile /tmp/run_coder_optimization.py
import asyncio
import json
import os
import shutil
import sys
import time
import yaml
from datetime import datetime

os.chdir('/content/drive/MyDrive/CodeAgent-MCP')
sys.path.insert(0, '.')

with open('/tmp/.api_key') as f:
    os.environ['OPENAI_API_KEY'] = f.read().strip()

WORKSPACE = '/tmp/workspace_optim'
os.makedirs(WORKSPACE, exist_ok=True)
os.environ['FILE_SERVER_ROOT'] = WORKSPACE
os.environ['SHELL_SERVER_CWD'] = WORKSPACE
os.environ['GIT_SERVER_ROOT'] = WORKSPACE

from eval.run_eval import load_benchmark, run_single_task
from src.core.config import load_settings, load_agents_config, load_mcp_config

SELECTED_TASKS = ['B1', 'B5', 'B6']

OPTIMIZED_PROMPT = '''你是一个高级Python开发工程师。根据任务描述编写高质量的 Python 代码，包含类型注解，遵循 PEP 8。

当你有工具可用时，严格按以下顺序操作：
第1步：如果 workspace 有已有文件，file_read 需要参考的文件（每个文件只读一次）
第2步：用 file_write 写入主源码文件（一次调用写完整个文件）
第3步：用 file_write 写入测试文件
第4步（可选）：用 shell_exec 运行 pytest 验证

工具调用效率规则：
- 不要在空目录调用 file_list 或 file_search
- 不要写完文件后再 file_read 自己刚写的文件
- 不要多次小片段追加写入，一次 file_write 写完整个文件
- 不要重复读取同一个文件，读一次后记住内容
- 不要用 file_search 搜索你刚创建的文件
- 已知 workspace 为空时，直接跳到 file_write

完成后用一句话说明做了什么，不要重复输出代码内容。
'''


def clean_workspace():
    for item in os.listdir(WORKSPACE):
        p = os.path.join(WORKSPACE, item)
        if os.path.isdir(p):
            shutil.rmtree(p, ignore_errors=True)
        else:
            os.remove(p)


async def run_with_config(tasks, config_name, agents_config):
    settings = load_settings()
    mcp_config = load_mcp_config()
    results = []
    for t in tasks:
        clean_workspace()
        print(f'  Running {t["task_id"]}: {t["name"]}...')
        r = await run_single_task(t, settings, agents_config, mcp_config, 'default', use_mcp=True)
        ws_files = [f for f in os.listdir(WORKSPACE) if os.path.isfile(os.path.join(WORKSPACE, f))]
        r['workspace_files'] = ws_files
        results.append(r)
        print(f'    -> score={r.get("review_score", "N/A")}, tokens={r.get("total_tokens", "?")}, files={ws_files}')
    return results


async def main():
    all_tasks = load_benchmark()
    tasks = [t for t in all_tasks if t['task_id'] in SELECTED_TASKS]
    print(f'Tasks: {[t["task_id"] for t in tasks]}')

    agents_config = load_agents_config()
    original_prompt = agents_config['coder']['system_prompt']

    # --- Baseline ---
    print('\n' + '='*60)
    print('Config: baseline (current prompt)')
    print('='*60)
    baseline_results = await run_with_config(tasks, 'baseline', agents_config)

    # --- Optimized ---
    print('\n' + '='*60)
    print('Config: optimized (efficiency rules)')
    print('='*60)
    agents_config['coder']['system_prompt'] = OPTIMIZED_PROMPT
    optimized_results = await run_with_config(tasks, 'optimized', agents_config)

    # --- Restore ---
    agents_config['coder']['system_prompt'] = original_prompt

    # --- Compare ---
    print('\n' + '='*60)
    print('CODER OPTIMIZATION COMPARISON')
    print('='*60)
    print(f'{"Task":<12} {"Baseline Tok":<15} {"Optimized Tok":<15} {"Reduction":<12} {"Score B/O":<12}')
    print('-' * 70)

    total_baseline = 0
    total_optimized = 0
    for i, t in enumerate(tasks):
        b_tok = baseline_results[i].get('total_tokens', 0)
        o_tok = optimized_results[i].get('total_tokens', 0)
        total_baseline += b_tok
        total_optimized += o_tok
        reduction = f'{(1 - o_tok/b_tok)*100:.0f}%' if b_tok > 0 else 'N/A'
        b_score = baseline_results[i].get('review_score', '-')
        o_score = optimized_results[i].get('review_score', '-')
        print(f'{t["task_id"]:<12} {b_tok:<15} {o_tok:<15} {reduction:<12} {b_score}/{o_score}')

    total_reduction = f'{(1 - total_optimized/total_baseline)*100:.0f}%' if total_baseline > 0 else 'N/A'
    print(f'{"TOTAL":<12} {total_baseline:<15} {total_optimized:<15} {total_reduction}')

    # Save
    os.makedirs('eval/results', exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    path = f'eval/results/coder_optimization_{timestamp}.json'
    with open(path, 'w', encoding='utf-8') as f:
        json.dump({
            'experiment': 'coder_optimization',
            'timestamp': timestamp,
            'tasks': SELECTED_TASKS,
            'baseline': {'total_tokens': total_baseline, 'results': baseline_results},
            'optimized': {'total_tokens': total_optimized, 'results': optimized_results},
            'token_reduction': total_reduction,
        }, f, indent=2, ensure_ascii=False)
    print(f'\nResults saved to: {path}')

asyncio.run(main())

In [ ]:
# Cell 9: 运行优化对比实验
# 预估耗时 ~20-30 分钟 (6 次 MCP 运行)
!python /tmp/run_coder_optimization.py

---
## P3: 多模型对比

在 no-mcp benchmark (8 tasks) 上对比 DeepSeek-chat vs Qwen2.5-72B (SiliconFlow)。

**前提**: 需要在 Colab Secrets 中配置 `SILICONFLOW_API_KEY`。
如果没有 SiliconFlow key，可以跳过此实验。

In [ ]:
# Cell 10: 多模型对比脚本
%%writefile /tmp/run_model_comparison.py
import asyncio
import json
import os
import sys
import time
from datetime import datetime

os.chdir('/content/drive/MyDrive/CodeAgent-MCP')
sys.path.insert(0, '.')

with open('/tmp/.api_key') as f:
    os.environ['OPENAI_API_KEY'] = f.read().strip()

sf_key = os.environ.get('SILICONFLOW_API_KEY', '')
if not sf_key:
    print('ERROR: SILICONFLOW_API_KEY not set. Skipping.')
    sys.exit(0)

from eval.run_eval import load_benchmark, run_single_task, print_summary
from src.core.config import load_settings, load_agents_config, load_mcp_config


async def main():
    tasks = load_benchmark()
    settings = load_settings()
    agents_config = load_agents_config()
    mcp_config = load_mcp_config()

    # Override siliconflow provider API key
    settings['providers']['siliconflow']['api_key'] = sf_key

    all_results = {}

    for provider in ['default', 'siliconflow']:
        model = settings['providers'][provider]['model']
        print(f'\n{"="*60}')
        print(f'Provider: {provider} ({model})')
        print(f'{"="*60}')

        results = []
        for i, task in enumerate(tasks):
            print(f'  [{i+1}/{len(tasks)}] {task["task_id"]}: {task["name"]}...')
            r = await run_single_task(task, settings, agents_config, mcp_config, provider, use_mcp=False)
            results.append(r)
            print(f'    -> {r["status"]}, score={r.get("review_score", "N/A")}, tokens={r.get("total_tokens", "?")}')

        all_results[provider] = results
        print_summary(results)

    # Compare
    print(f'\n{"="*60}')
    print('MODEL COMPARISON')
    print(f'{"="*60}')
    print(f'{"Task":<20} {"DeepSeek":<20} {"Qwen2.5-72B":<20}')
    print('-' * 60)
    for i, task in enumerate(tasks):
        d = all_results['default'][i]
        s = all_results['siliconflow'][i]
        d_info = f's={d.get("review_score","-")} t={d.get("total_tokens",0)}'
        s_info = f's={s.get("review_score","-")} t={s.get("total_tokens",0)}'
        print(f'{task["task_id"]:<20} {d_info:<20} {s_info:<20}')

    for name, results in all_results.items():
        scores = [r.get('review_score', 0) for r in results if r.get('review_score')]
        tokens = [r.get('total_tokens', 0) for r in results]
        avg_s = sum(scores)/len(scores) if scores else 0
        total_t = sum(tokens)
        model = settings['providers'][name]['model']
        print(f'\n{model}: avg_score={avg_s:.1f}, total_tokens={total_t:,}')

    # Save
    os.makedirs('eval/results', exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output = {'experiment': 'model_comparison', 'timestamp': timestamp, 'providers': {}}
    for name, results in all_results.items():
        scores = [r.get('review_score', 0) for r in results if r.get('review_score')]
        tokens = [r.get('total_tokens', 0) for r in results]
        output['providers'][name] = {
            'model': settings['providers'][name]['model'],
            'avg_score': round(sum(scores)/len(scores), 1) if scores else 0,
            'total_tokens': sum(tokens),
            'results': results,
        }
    path = f'eval/results/model_comparison_{timestamp}.json'
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    print(f'\nResults saved to: {path}')

asyncio.run(main())

In [ ]:
# Cell 11: 运行多模型对比
# 预估耗时 ~30 分钟 (16 次运行: 2 providers × 8 tasks)
# 如果没有 SiliconFlow key，此 cell 会自动跳过
!python /tmp/run_model_comparison.py

---
## 汇总所有增强实验结果

In [ ]:
# Cell 12: 汇总
import json
import os

results_dir = 'eval/results'

def load_latest(prefix):
    files = sorted([f for f in os.listdir(results_dir) if f.startswith(prefix) and f.endswith('.json')])
    if files:
        with open(os.path.join(results_dir, files[-1]), encoding='utf-8') as f:
            return json.load(f), files[-1]
    return None, None

print('='*70)
print('ENHANCEMENT EXPERIMENTS SUMMARY')
print('='*70)

# P1: HumanEval
he, he_file = load_latest('humaneval_')
if he:
    s = he['summary']
    print(f'\n--- P1: HumanEval ({he_file}) ---')
    print(f'  pass@1: {s["passed"]}/{s["total"]} ({s["pass_at_1"]:.1%})')
    print(f'  Time: {s.get("elapsed_seconds", "?")}s')

# P2: Coder optimization
co, co_file = load_latest('coder_optimization_')
if co:
    print(f'\n--- P2: Coder Optimization ({co_file}) ---')
    print(f'  Baseline tokens: {co["baseline"]["total_tokens"]:,}')
    print(f'  Optimized tokens: {co["optimized"]["total_tokens"]:,}')
    print(f'  Reduction: {co["token_reduction"]}')

# P3: Model comparison
mc, mc_file = load_latest('model_comparison_')
if mc:
    print(f'\n--- P3: Model Comparison ({mc_file}) ---')
    for name, data in mc['providers'].items():
        print(f'  {data["model"]}: avg_score={data["avg_score"]}, total_tokens={data["total_tokens"]:,}')

print('\n' + '='*70)
print('Copy these results to update README.md and INTERVIEW_PREP.md')
print('='*70)

---
## 运行指南

1. **Cell 1-2**: 环境准备 + API Keys
2. **Cell 3-6**: P1 HumanEval (~30-40 min)
   - Cell 5 先跑 20 题验证
   - Cell 6 跑全量 164 题
3. **Cell 7-9**: P2 Coder 优化对比 (~20-30 min)
4. **Cell 10-11**: P3 多模型对比 (~30 min, 需要 SiliconFlow key)
5. **Cell 12**: 汇总结果

所有结果自动保存到 `eval/results/` 目录。